# Intermediate: Recurrent Neural Networks (RNN)

This notebook covers:
- RNN fundamentals and architecture
- LSTM (Long Short-Term Memory)
- GRU (Gated Recurrent Unit)
- Sequence modeling
- Text classification
- Time series prediction

## Why RNNs?

RNNs are designed for sequential data:
- Text (words in a sentence)
- Time series (stock prices)
- Audio (sound waves)
- Video (frames)

```
Traditional NN:        RNN:
Input → Output        Input₁ → State₁ → Output₁
                      Input₂ → State₂ → Output₂
                      Input₃ → State₃ → Output₃
                          ↑       ↑
                      (Memory of previous inputs)
```


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Simple RNN from Scratch

Understanding the basic RNN cell:

```
     h(t-1)
       ↓
    ┌─────┐
x(t)→│ RNN │→ h(t)
    └─────┘
       ↓
     y(t)

h(t) = tanh(W_hh * h(t-1) + W_xh * x(t) + b_h)
y(t) = W_hy * h(t) + b_y
```


In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        
        # RNN weights
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, hidden):
        # x: (batch, input_size)
        # hidden: (batch, hidden_size)
        combined = torch.cat((x, hidden), 1)
        hidden = torch.tanh(self.i2h(combined))
        output = self.h2o(hidden)
        return output, hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)

# Test the simple RNN
input_size = 10
hidden_size = 20
output_size = 5
batch_size = 3

rnn = SimpleRNN(input_size, hidden_size, output_size).to(device)
x = torch.randn(batch_size, input_size).to(device)
hidden = rnn.init_hidden(batch_size)

output, hidden = rnn(x, hidden)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Hidden shape: {hidden.shape}")


## 2. PyTorch RNN Layers

PyTorch provides built-in RNN layers.


In [ ]:
# nn.RNN
rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

# Input: (batch, sequence_length, input_size)
x = torch.randn(3, 5, 10)  # batch=3, seq_len=5, features=10

# Forward pass
output, hidden = rnn(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")  # (batch, seq_len, hidden_size)
print(f"Hidden shape: {hidden.shape}")  # (num_layers, batch, hidden_size)


## 3. LSTM (Long Short-Term Memory)

LSTM solves the vanishing gradient problem in vanilla RNNs.

```
LSTM Cell:
        ┌─────────────────────┐
        │   Forget Gate (f)   │
        │   Input Gate (i)    │
   x(t) │   Output Gate (o)   │ h(t)
   ───→ │   Cell Gate (g)     │ ───→
        │                     │
        │   Cell State c(t)   │
        └─────────────────────┘

f = σ(W_f·[h(t-1), x(t)] + b_f)    # What to forget
i = σ(W_i·[h(t-1), x(t)] + b_i)    # What to remember
o = σ(W_o·[h(t-1), x(t)] + b_o)    # What to output
g = tanh(W_g·[h(t-1), x(t)] + b_g) # New candidates

c(t) = f ⊙ c(t-1) + i ⊙ g          # Update cell state
h(t) = o ⊙ tanh(c(t))              # Output
```


In [ ]:
# LSTM
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

x = torch.randn(3, 5, 10)  # (batch, seq_len, input_size)

# LSTM returns output and (hidden, cell) states
output, (hidden, cell) = lstm(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Hidden shape: {hidden.shape}")
print(f"Cell shape: {cell.shape}")


## 4. Text Classification with LSTM

Let's build a sentiment classifier.


In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout=0.5):
        super(TextClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers, 
                           batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        # text: (batch, seq_len)
        embedded = self.dropout(self.embedding(text))
        # embedded: (batch, seq_len, embedding_dim)
        
        output, (hidden, cell) = self.lstm(embedded)
        # output: (batch, seq_len, hidden_dim)
        # hidden: (n_layers, batch, hidden_dim)
        
        # Use the last hidden state
        hidden = self.dropout(hidden[-1])
        # hidden: (batch, hidden_dim)
        
        return self.fc(hidden)

# Create model
vocab_size = 5000
embedding_dim = 100
hidden_dim = 256
output_dim = 2  # Binary classification
n_layers = 2

model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers).to(device)
print(model)

# Test forward pass
batch_size = 4
seq_len = 50
text = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
output = model(text)
print(f"\nInput shape: {text.shape}")
print(f"Output shape: {output.shape}")


## 5. GRU (Gated Recurrent Unit)

GRU is a simpler alternative to LSTM with fewer parameters.

```
GRU Cell (simpler than LSTM):
        ┌─────────────────┐
        │  Reset Gate (r) │
   x(t) │  Update Gate(z) │ h(t)
   ───→ │  New State (n)  │ ───→
        └─────────────────┘

r = σ(W_r·[h(t-1), x(t)])
z = σ(W_z·[h(t-1), x(t)])
n = tanh(W_n·[r⊙h(t-1), x(t)])
h(t) = (1-z)⊙n + z⊙h(t-1)
```


In [ ]:
# GRU
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

x = torch.randn(3, 5, 10)
output, hidden = gru(x)

print(f"GRU Output shape: {output.shape}")
print(f"GRU Hidden shape: {hidden.shape}")

# Compare parameters
lstm_params = sum(p.numel() for p in lstm.parameters())
gru_params = sum(p.numel() for p in gru.parameters())
print(f"\nLSTM parameters: {lstm_params:,}")
print(f"GRU parameters: {gru_params:,}")
print(f"GRU has {(1 - gru_params/lstm_params)*100:.1f}% fewer parameters")


## 6. Bidirectional RNN

Process sequences in both directions for better context.

```
Forward:  x₁ → x₂ → x₃ → x₄
          ↓    ↓    ↓    ↓
Backward: x₁ ← x₂ ← x₃ ← x₄
          
Output = [forward_hidden; backward_hidden]
```


In [ ]:
# Bidirectional LSTM
bi_lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=1, 
                 batch_first=True, bidirectional=True)

x = torch.randn(3, 5, 10)
output, (hidden, cell) = bi_lstm(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")  # hidden_size * 2
print(f"Hidden shape: {hidden.shape}")  # num_layers * 2

# For classification, concatenate final forward and backward states
forward_hidden = hidden[0, :, :]
backward_hidden = hidden[1, :, :]
final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
print(f"\nFinal hidden shape: {final_hidden.shape}")


## 7. Sequence-to-Sequence Model

For tasks like machine translation.

```
Encoder-Decoder:

Input Sequence:    x₁  x₂  x₃  <EOS>
                   ↓   ↓   ↓    ↓
Encoder:         [RNN][RNN][RNN][RNN] → Context Vector
                                         ↓
Decoder:                        [RNN][RNN][RNN]
                                  ↓    ↓    ↓
Output Sequence:                 y₁   y₂   y₃
```


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src: (seq_len, batch)
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        # input: (batch)
        input = input.unsqueeze(0)  # (1, batch)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (seq_len, batch)
        # trg: (seq_len, batch)
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features
        
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device)
        hidden, cell = self.encoder(src)
        
        input = trg[0, :]  # First input to decoder is <SOS>
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
            
        return outputs

# Create Seq2Seq model
INPUT_DIM = 1000
OUTPUT_DIM = 1000
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder).to(device)

print(f"Seq2Seq model created")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 8. Practical Tips for RNNs

### Common Issues and Solutions

1. **Vanishing/Exploding Gradients**
   - Use LSTM or GRU instead of vanilla RNN
   - Apply gradient clipping
   - Use skip connections

2. **Slow Training**
   - Use GRU instead of LSTM (fewer parameters)
   - Reduce sequence length
   - Use packed sequences for variable lengths

3. **Poor Performance**
   - Use bidirectional RNNs
   - Add attention mechanisms
   - Increase model capacity
   - Use pretrained embeddings


In [ ]:
# Gradient clipping
def train_with_clipping(model, data, optimizer, clip):
    model.train()
    epoch_loss = 0
    
    for batch in data:
        optimizer.zero_grad()
        output = model(batch)
        loss = criterion(output, target)
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    return epoch_loss / len(data)

# Packed sequences for variable length
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

def forward_with_packing(self, text, text_lengths):
    embedded = self.embedding(text)
    
    # Pack the sequence
    packed_embedded = pack_padded_sequence(embedded, text_lengths.cpu(), 
                                          batch_first=True, enforce_sorted=False)
    
    packed_output, (hidden, cell) = self.lstm(packed_embedded)
    
    # Unpack
    output, output_lengths = pad_packed_sequence(packed_output, batch_first=True)
    
    return self.fc(hidden[-1])

print("Training tips demonstrated!")


## 📝 Summary

✅ Understood RNN architecture and mechanics  
✅ Implemented LSTM and GRU networks  
✅ Built text classifiers  
✅ Created sequence-to-sequence models  
✅ Learned bidirectional RNNs  
✅ Applied best practices  

### Key Takeaways

1. **Use LSTM/GRU** for most tasks (not vanilla RNN)
2. **Bidirectional** often improves performance
3. **Gradient clipping** is essential
4. **Packed sequences** for variable-length inputs
5. **Teacher forcing** helps training seq2seq models

### Next Steps

- **Next**: `03_advanced_training.ipynb`
- **Practice**: Build a text generator
- **Challenge**: Implement attention mechanism


## 🎯 Practice Exercises


In [ ]:
# Exercise 1: Build a character-level RNN for name generation
# Your code here:


# Exercise 2: Implement a simple machine translation model
# Your code here:


# Exercise 3: Create a time series predictor with LSTM
# Your code here:


# Exercise 4: Build a sentiment analyzer with bidirectional GRU
# Your code here:
